# 1、将原始keywords_train/test.jsonl做格式转换

In [2]:
from datasets import load_dataset
# 1.1 加载数据
data = load_dataset("json",data_files={"train":"./data/keywords_data_train.jsonl","test":"./data/keywords_data_test.jsonl"})

In [6]:
data["train"][0]
data["train"].column_names

['conversation_id', 'category', 'conversation', 'dataset']

In [ ]:
# 1.2 将数据转换成SFTTrainer所需要的 Language Modeling 这种类型，对话格式的数据
def convert_func(examples:dict[str, list]):
    """
    接收的参数，就是.map方法传递的，原始的数据，以批次形式接收
    """
    conversation_lists: list[list] =examples["conversation"]
    messages_lists:list[list] = []
    for conversation in conversation_lists:
        # conversation是单条样本所对应的列表：
        human_message = conversation[0]["human"]
        assistant_message = conversation[0]["assistant"]
        message_list = [ 
            {"role":"user","content":human_message},
            {"role":"assistant","content":assistant_message}
        ]
        messages_lists.append(message_list)

    return {"messages":messages_lists}



converted_data = data.map(convert_func,batched=True,remove_columns=data["train"].column_names,)

In [11]:
converted_data

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 49500
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 500
    })
})

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("model/Qwen3-0.6B/")
def convert_func(examples:dict[str, list]):
    """
    接收的参数，就是.map方法传递的，原始的数据，以批次形式接收
    """
    message_lists = examples["messages"]

    all_length_list = []
    for message_list in message_lists:

        token_id_list = tokenizer.apply_chat_template(message_list)["input_ids"]

        all_length_list.append(len(token_id_list))

    return {"length":all_length_list}

data_with_length = converted_data.map(convert_func,batched=True,remove_columns="messages")

Map: 100%|██████████| 500/500 [00:00<00:00, 2771.18 examples/s]


In [8]:
data_frame = data_with_length["train"].to_pandas()


In [10]:
data_frame["length"].quantile(0.9999999)

np.float64(1182.8515030001727)

In [10]:
converted_data["train"][0]

{'messages': [{'content': '高氟铍矿石在熔炼过程中配入氢氧化铝来脱除其中的氟.结果表明,在配入5％Na2CO3、9.3％Al(OH)3、1400～1500℃熔炼20 min的情况下,BeO回收率达到96％以上,脱氟效果良好(铍玻璃F/BeO能控制在15％以内).为高氟铍矿石的工业应用探索出新的冶炼途径.\n找出上文中的关键词',
   'role': 'user'},
  {'content': '高氟铍矿;配料;熔炼;回收率;脱氟率', 'role': 'assistant'}]}

# 2、构造SFTConfig对象（需要理解SFTConfig有哪些重点参数）

In [19]:
from trl.trainer.sft_config import SFTConfig
import os
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs/04_trl_sft_demo"
config = SFTConfig(
    # 数据规模相关的
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps= 32,
    max_steps=500,
    # num_train_epochs= # max_steps会比num_train_epochs的优先级更高
    # 训练可视化相关
    logging_strategy="steps",
    logging_steps=25,
    report_to="tensorboard", # 要想去进一步制定tensorboard 日志文件保存位置，需要通过os.environ去指定,
    # 学习率和优化器相关
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_steps= 0.1,
    # optim="" 优化器的类型，默认值就是adamW
    # 评估和保存相关
    eval_strategy="steps",
    eval_steps=50,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    output_dir="./finetuned/04_trl_sft_demo", # 保存的是检查点
    bf16=True,
    gradient_checkpointing=False,
    activation_offloading=False,
    max_length=700,
    # 原生的qwen3的聊天模板，和assistant_only_loss参数不兼容，所以需要基于原生的chat_template进行修改，得到new_chat_template.jinja文件
    # 可以通过chat_template_path传递新的chat_template文件
    assistant_only_loss=True,
    chat_template_path="./new_chat_template.jinja"
)

# 3、基于SFTConfig，数据集，模型，tokenizer等，去构建一个SFTTrainer实例

In [20]:
from transformers import AutoModelForCausalLM
from trl.trainer.sft_trainer import SFTTrainer
from tyro import conf
model = AutoModelForCausalLM.from_pretrained("model/Qwen3-0.6B/")

trainer = SFTTrainer(
    model=model,
    args=config,
    train_dataset=converted_data["train"],
    eval_dataset=converted_data["test"],
    # processing_class指的就是tokenizer参数
    processing_class= tokenizer
)

Loading weights:   1%|▏         | 4/311 [00:00<00:00, 799.83it/s, Materializing param=model.layers.0.mlp.down_proj.weight]  

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1370.77it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
# 对于数据的简单探测
dataloader = trainer.get_train_dataloader()
for batch in dataloader:
    input_ids = batch["input_ids"]
    result = tokenizer.decode(input_ids[0])
    print(result)
    break

<|im_start|>user
关键词抽取：
以手动换挡机构疲劳寿命试验为目的,构建了一种模拟驾驶员进行选档、换挡操作的试验平台,该平台集成二自由度运动滑台与气动加载装置为一体形成换挡运动加载机构.以该机构为研究对象,通过建立换挡与选挡的运动轨迹模型,分析换挡运动加载机构位移输出与运动轨迹之间的关系,通过分析换挡机构操纵杆受力情况,分别对换挡动作和选档动作进行力学分析,并得出加载力的计算方法.最后结合电气控制技术与气动控制技术,对系统进行了试验,结果表明,系统具有可行性与正确性.<|im_end|>
<|im_start|>assistant
<think>

</think>

换挡机构;试验平台;疲劳性能<|im_end|>

<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>


# 4、调用SFTTrainer.train()

In [21]:
# 封装了整个，训练，验证，保存完整的过程，可以传递resume_from_checkpoint，表示从某个检查点开始训练，不传表示从头开始
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,1.163262,1.120147,3.004588,283650.000000,0.754084


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.90s/it]


KeyboardInterrupt: 

# 5、调用trainer.save_model()把model,tokenzier保存

In [ ]:
trainer.save_model("./finetuned/04_trl_sft_demo")